# LLM Process

A notebook used to simplify (ie. remove columns from & restructure) the dataset for LLM processing

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 

## Setup

In [2]:
data_dir = "../raw"
# filename = data_dir + "/procedure_jan_2015_jan_2021.csv"
filename = data_dir + "/humanfactors_jan_2020_jan_2021.csv"
raw_df = df = pd.read_csv(filename, index_col=0, header=[0, 1], low_memory=False)

In [34]:
subs

['Other / Unknown',
 'Confusion',
 ' Situational Awareness',
 'Distraction',
 ' Situational Awareness',
 'Communication Breakdown',
 'Fatigue',
 ' Situational Awareness']

In [47]:
from sklearn.preprocessing import MultiLabelBinarizer

def normalize(li:list[str]):
    if (not isinstance(li, list)) and pd.isnull(li):
        return []
    return [s.lower().strip() for s in li]

MLB = MultiLabelBinarizer()
subs = raw_df['Person 1']['Human Factors'].str.split(";").tolist()

subs = [normalize(li) for li in subs]
bins = pd.DataFrame(MLB.fit_transform(subs), columns=MLB.classes_)
bins

,communication breakdown,confusion,distraction,fatigue,human-machine interface,other / unknown,physiological - other,situational awareness,time pressure,training / qualification,troubleshooting,workload
0,0,0,0,0,0,1,0,0,0,0,0,0
1,0,1,0,0,0,0,0,1,0,0,0,0
2,0,0,1,0,0,0,0,1,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,1,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
2178,0,1,0,0,0,0,0,0,0,0,0,0
2179,0,0,1,0,0,0,0,1,0,1,0,0
2180,0,0,0,0,0,0,0,1,0,0,0,0
2181,1,1,1,0,1,0,0,1,1,1,0,1


In [49]:
for col in bins.columns:
    display(bins[col].value_counts())

communication breakdown
0    1238
1     945
Name: count, dtype: int64

confusion
0    1792
1     391
Name: count, dtype: int64

distraction
0    1594
1     589
Name: count, dtype: int64

fatigue
0    2121
1      62
Name: count, dtype: int64

human-machine interface
0    2009
1     174
Name: count, dtype: int64

...

situational awareness
1    1093
0    1090
Name: count, dtype: int64

time pressure
0    1920
1     263
Name: count, dtype: int64

training / qualification
0    1883
1     300
Name: count, dtype: int64

troubleshooting
0    2092
1      91
Name: count, dtype: int64

workload
0    1891
1     292
Name: count, dtype: int64

In [32]:
flat = raw_df.copy()
flat.columns = [f"{a.replace(' ', "_")}_{b.replace(' ', "_")}" for a, b in flat.columns]
flat.head()

,Time_Date,Time_Local_Time_Of_Day,Place_Locale_Reference,Place_State_Reference,Place_Relative_Position.Angle.Radial,Place_Relative_Position.Distance.Nautical_Miles,Place_Altitude.AGL.Single_Value,Place_Altitude.MSL.Single_Value,Place_Latitude_/_Longitude_(UAS),Environment_Flight_Conditions,...,Events_When_Detected,Events_Result,Assessments_Contributing_Factors_/_Situations,Assessments_Primary_Problem,Report_1_Narrative,Report_1_Callback,Report_2_Narrative,Report_2_Callback,Report_1_Synopsis,Unnamed:_125_level_0_Unnamed:_125_level_1
1715004,202001,1201-1800,ZZZ.Airport,US,NaN,NaN,0.0,NaN,NaN,NaN,...,In-flight,Flight Crew Landed in Emergency Condition; Fli...,Human Factors; Logbook Entry,Human Factors,During preflight planning discovered this was ...,NaN,Flight departing ZZZ to ZZZZ. Aircraft changed...,NaN,B777-200 reported returning to departure airpo...,NaN
1715238,202001,0001-0600,L30.TRACON,NV,NaN,NaN,NaN,5000.0,NaN,VMC,...,In-flight,Air Traffic Control Issued New Clearance,Airspace Structure; Human Factors,Human Factors,Aircraft X was inbound to the airport from the...,NaN,NaN,NaN,L30 TRACON Controller reported they issued VFR...,NaN
1715252,202001,1201-1800,DCA.Tower,DC,NaN,NaN,NaN,NaN,NaN,NaN,...,In-flight,Air Traffic Control Issued New Clearance; Flig...,Human Factors; Weather,Human Factors,I was working local control in a south operati...,NaN,NaN,NaN,DCA Tower Controller realized a plan was not g...,NaN
1715281,202001,1201-1800,TPA.Airport,FL,NaN,3.0,0.0,NaN,NaN,NaN,...,Taxi,General None Reported / Taken,Human Factors,Human Factors,We flew a long pattern to Runway 1L arriving f...,NaN,NaN,NaN,B737 Captain reported landing without a cleara...,NaN
1715404,202001,1201-1800,BFI.Airport,US,NaN,NaN,NaN,NaN,NaN,VMC,...,Taxi,Air Traffic Control Issued New Clearance; Flig...,Human Factors,Human Factors,I landed without clearance. We landed on Appro...,NaN,NaN,NaN,Captain reported landing without clearance; ci...,NaN


In [33]:
# print(flat.iloc[0].to_json())
tt = flat[flat.Assessments_Primary_Problem == 'Human Factors']
with pd.option_context(
    "display.max_rows", None,
):  # more options can be specified also
    print(

    tt[[
        "Assessments_Primary_Problem",
        "Person_1_Human_Factors",
        # "Person_2_Human_Factors",
        # "",
    ]].map(lambda x : str(set(x.lower().split("; "))) if isinstance(x, str) else "NaN").value_counts()
    )

Assessments_Primary_Problem  Person_1_Human_Factors                                                                                                                                              
{'human factors'}            {'communication breakdown'}                                                                                                                                             244
                             NaN                                                                                                                                                                     233
                             {'situational awareness'}                                                                                                                                               194
                             {'communication breakdown', 'situational awareness'}                                                                                                                    131
  

In [34]:
narrative_only = flat[
    ["Assessments_Primary_Problem", "Report_1_Narrative", "Report_2_Narrative"]
]

### Cleaning Target & De-duplication
Here we examine if the target is NaN or dataset has any duplicates. Both Report_1_Narrative & Report_2_Narrative have duplicates (which shouldn't be the case).

In [35]:
narrative_only.info()

<class 'pandas.DataFrame'>
Index: 2183 entries, 1715004 to 2222274
Data columns (total 3 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   Assessments_Primary_Problem  2183 non-null   str  
 1   Report_1_Narrative           2183 non-null   str  
 2   Report_2_Narrative           420 non-null    str  
dtypes: str(3)
memory usage: 68.2 KB


In [36]:
narrative_only.describe()

,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative
count,2183,2183,420
unique,1,2183,388
top,Human Factors,During preflight planning discovered this was ...,[Report narrative contained no additional info...
freq,2183,1,25


In [37]:
# Identify Report 1 Narrative that are duplicated

r1_dupes = narrative_only[
    narrative_only.Report_1_Narrative.duplicated(keep=False)
]
r1_dupes.dropna(how="all")

,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative


In [38]:
r2_only = narrative_only.copy()
r2_dupes = r2_only[r2_only.Report_2_Narrative.duplicated(keep=False) & pd.notna(r2_only.Report_2_Narrative)]
r2_dupes.dropna(how="all")
display(r2_dupes.Report_2_Narrative.value_counts())
print("Sentinel values have lots of duplicates.")


Report_2_Narrative
[Report narrative contained no additional information.]    25
[Report narrative contained no additional information]      5
[Report narrative contained no additional information].     4
[Narrative contained no additional information.]            2
Name: count, dtype: int64

Sentinel values have lots of duplicates.


In [39]:
# Find all occurrences of sentinel value string to ensure we don't delete actual data
# r2_no_sentinel = []
sentinel = "contained no additional information"
print("R1 Narrative w/ sentinels")
display(narrative_only[
    narrative_only.Report_1_Narrative.str.contains(sentinel)
])
print("R2 Narrative w/ sentinels")
display(narrative_only[narrative_only.Report_2_Narrative.str.contains(sentinel)][
    ["Report_2_Narrative"]
].value_counts())
print("So we can safely replace sentinel string w/ NaN & not delete real info.")

R1 Narrative w/ sentinels


,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative
1761723,Human Factors,[Report narrative contained no additional info...,The crew and plane had arrived 4 hours earlier...


R2 Narrative w/ sentinels


Report_2_Narrative                                     
[Report narrative contained no additional information.]    25
[Report narrative contained no additional information]      5
[Report narrative contained no additional information].     4
[Narrative contained no additional information.]            2
Report contained no additional information.                 1
[Report contained no additional information.]               1
Report narrative contained no additional information.       1
Name: count, dtype: int64

So we can safely replace sentinel string w/ NaN & not delete real info.


In [40]:
no_nan_r2_dupes = r2_dupes[~r2_dupes.Report_2_Narrative.str.lower().str.contains(sentinel)]
display(no_nan_r2_dupes.sort_values(by="Report_2_Narrative"))
no_nan_r2_dupes.sort_values(by="Report_2_Narrative").index


,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative


Index([], dtype='int64')

After referencing the DB, we can: 
1) Drop all items where the primary problem is NaN (since there is no label available)
2) Deduplicate & keep the first Report_1_Narrative (indeed, the narratives are exactly the same and in ONE case -- ACN 2233790/2242665 -- they have different primary problem labels)
3) Replace sentinel values (eg. "contained not additional info") with NaN
4) Apply Keep-first-override to all the duplicate Report_2_Narratives (see README) & null Report_1_Narratives

In [42]:
sentinel_substring = "contained no additional information"
sentinel_substring_2 = "narrative added no additional information"
sentinel_substring_3 = "narrative contains no additional information"

sentinels = [sentinel_substring, sentinel_substring_2, sentinel_substring_3]

def convert_sentinel(s:str):
    if not isinstance(s, str) or any([phrase in s.lower() for phrase in sentinels]):
        return np.nan
    
    return s


filtered = narrative_only[pd.notna(narrative_only.Assessments_Primary_Problem)].copy()
filtered["Report_1_Narrative"] = filtered.Report_1_Narrative.apply(convert_sentinel)
filtered = filtered[pd.notna(filtered['Report_1_Narrative'])]
filtered = filtered.drop_duplicates(["Report_1_Narrative"], keep="first")
filtered["Report_2_Narrative"] = filtered.Report_2_Narrative.apply(convert_sentinel)

filtered.describe()

,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative
count,2182,2182,380
unique,1,2182,380
top,Human Factors,During preflight planning discovered this was ...,Flight departing ZZZ to ZZZZ. Aircraft changed...
freq,2182,1,1


## Normalize Target
Now we make sure the Primary Problem is just one of the major categories (not subcategories).


In [44]:
category_counts = filtered.Assessments_Primary_Problem.value_counts()
print(category_counts.to_string())

Assessments_Primary_Problem
Human Factors    2182


In [45]:
s = 0
for c, r in zip(category_counts.index, category_counts):
    li = c.replace(" ", "").split(";") #bc sometimes we have things like 'aircraft; aircraft'
    if len(set(li)) > 1:
        s+= r
    
print(s, "records are a multi-class target")
print(round(s / category_counts.sum(), 4), "of all records")

0 records are a multi-class target
0.0 of all records


In [46]:
def setify(x:str):
    return set(x.lower().replace(" ", "").split(";"))


processed = filtered.copy()
processed["setify"] = processed.Assessments_Primary_Problem.apply(setify)
processed['is_multi'] = processed.setify.apply(lambda x : len(x) > 1)
processed = processed[~processed.is_multi]
processed["Assessments_Primary_Problem"] = processed.setify.apply(lambda x : list(x)[0])

display(processed.Assessments_Primary_Problem.value_counts())
processed.drop(['setify', 'is_multi'], axis=1, inplace=True)
processed.head()

Assessments_Primary_Problem
humanfactors    2182
Name: count, dtype: int64

,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative
1715004,humanfactors,During preflight planning discovered this was ...,Flight departing ZZZ to ZZZZ. Aircraft changed...
1715238,humanfactors,Aircraft X was inbound to the airport from the...,NaN
1715252,humanfactors,I was working local control in a south operati...,NaN
1715281,humanfactors,We flew a long pattern to Runway 1L arriving f...,NaN
1715404,humanfactors,I landed without clearance. We landed on Appro...,NaN


In [ ]:
processed.to_csv("2020_2021_human_factors.csv")

In [ ]:
# Count tokens, roughly
processed.Report_1_Narrative.apply(lambda x: x.count(" ")).sum()

np.int64(4843308)